# Введение

Этот ноутбук содержит финальную версию решения для хакатона **ChemAI: Predict the Cure**.

Наша задача - предсказать три показателя для химических соединений:

- `IC50` - концентрация, при которой вещество подавляет 50% активности вируса;
- `CC50` - концентрация, при которой вещество токсично для 50% клеток;
- `SI` - индекс селективности.

Финальный pipeline был собран после нескольких этапов работы команды.

Сначала Чевачин Глеб подготовил EDA в ноутбуке `01_eda.ipynb`: были проверены размеры данных, пропуски, статистики target-переменных, константные признаки, exact-match между train и test, медианный baseline и важная формула `SI = CC50 / IC50`.

После этого Рожков Артём сделал baseline-модели и первые рабочие submission. На этом этапе мы получили стартовую точку и поняли, что простых моделей недостаточно: особенно нестабильно предсказывался `SI`.

Дальше Арсений Сысоев и Евгений Полей занимались улучшением качества: проверяли разные варианты ансамблей, overlay, target-wise stack, работу с `SI`, а также финальные веса смешивания. Арсений также выполнял роль тимлида: собирал результаты, контролировал структуру решения и финальную версию pipeline.

Главная идея финального решения - не обучать одну модель на все target, а аккуратно собрать несколько устойчивых компонентов:

1. отдельно стабилизировать `SI`, так как он имеет сильный хвост;
2. использовать доменную связь `SI = CC50 / IC50`;
3. не пересчитывать `SI` на финальном этапе, если это ухудшает leaderboard;
4. улучшать `IC50` и `CC50` через дополнительные модели и небольшие overlay;
5. выбирать финальный submission по сочетанию Kaggle score и устойчивости распределений.

Финальная официальная версия submission:

```text
v34_adv_weighted_ic50_020_cc50_030_keep_si.csv
```

Официальный Kaggle score:

```text
269.39668
```

Логика финального смешивания:

```text
IC50 = 0.80 * base_IC50 + 0.20 * adv_IC50
CC50 = 0.70 * base_CC50 + 0.30 * adv_CC50
SI = base_SI
```

Здесь `base_SI` - уже стабилизированная версия `SI`, полученная через смешивание модельного предсказания и формулы `CC50 / IC50`.

В этом ноутбуке собран итоговый воспроизводимый pipeline: загрузка данных, подготовка признаков, обучение моделей, генерация промежуточных предсказаний, финальное смешивание и сохранение submission.


In [1]:
!pip install numpy pandas scikit-learn scipy xgboost lightgbm catboost

# 1. EDA


In [2]:
import warnings
from pathlib import Path
import random

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")


## 1. Подготовка


Базовое окружение для анализа собрано: подключены библиотеки для таблиц, массивов, путей и расчета RMSE. Предупреждения отключены, поэтому вывод ноутбука остается компактным.


In [3]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path(".")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

TARGETS_RAW = ["IC50, mM", "CC50, mM", "SI"]
TARGETS = ["IC50", "CC50", "SI"]
ID_COL = "index"


## 2. Конфигурация


Пути к данным, папка для результатов, список таргетов и seed заданы в одном месте. Это снижает риск ошибок в названиях колонок и делает запуск ноутбука воспроизводимым.


In [4]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def competition_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    scores = {
        "IC50": rmse(y_true[:, 0], y_pred[:, 0]),
        "CC50": rmse(y_true[:, 1], y_pred[:, 1]),
        "SI": rmse(y_true[:, 2], y_pred[:, 2]),
    }

    return float(np.mean(list(scores.values()))), scores


def make_row_hash(df):
    tmp = df.copy().fillna(-999999999)
    return pd.util.hash_pandas_object(tmp, index=False)


## 3. Вспомогательные функции


Метрика качества считается отдельно по IC50, CC50 и SI, а затем усредняется. Хеширование строк подготовлено для диагностики полных совпадений между train и test.


In [5]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = train.rename(columns={
    "IC50, mM": "IC50",
    "CC50, mM": "CC50",
})

feature_cols = [c for c in test.columns if c != ID_COL]

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGETS].astype(float).copy()
y_values = y.values

print("=" * 80)
print("DATA SHAPES")
print("=" * 80)
print("train:", train.shape)
print("test:", test.shape)
print("X:", X.shape)
print("X_test:", X_test.shape)
print("y:", y.shape)


DATA SHAPES
train: (751, 214)
test: (250, 211)
X: (751, 210)
X_test: (250, 210)
y: (751, 3)


## 4. Загрузка данных и формирование матриц


Данные успешно разделены на признаки train, признаки test и три целевые переменные. Размеры таблиц дают быстрый контроль, что чтение файлов и выбор колонок прошли корректно.


In [6]:
print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing_info = pd.DataFrame({
    "dataset": ["X_train", "X_test", "y"],
    "total_missing": [
        int(X.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
        int(y.isna().sum().sum()),
    ],
    "columns_with_missing": [
        int((X.isna().sum() > 0).sum()),
        int((X_test.isna().sum() > 0).sum()),
        int((y.isna().sum() > 0).sum()),
    ],
})

display(missing_info)



MISSING VALUES


,dataset,total_missing,columns_with_missing
0,X_train,24,12
1,X_test,12,12
2,y,0,0


## 5. Пропуски


Информация о пропусках собрана отдельно для обучающих признаков, тестовых признаков и таргетов. Этот результат показывает, нужна ли дополнительная обработка пустых значений перед моделированием.


In [7]:
print("\n" + "=" * 80)
print("TARGET STATISTICS")
print("=" * 80)

target_stats = y.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
display(target_stats)



TARGET STATISTICS


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
IC50,751.0,204.544021,370.367937,0.003517,0.043606,1.329394,13.222351,44.069306,206.787402,1002.304418,1334.274334,4095.188563
CC50,751.0,577.426098,641.515163,0.700808,0.845548,9.834099,99.998894,376.580899,877.508784,1883.480370,3162.007168,4538.976189
SI,751.0,89.153313,788.882198,0.011489,0.229697,0.941805,1.500000,4.000000,17.372463,138.433453,1281.250000,15620.600000


## 6. Статистика таргетов


Распределения целевых переменных выглядят сложными для моделирования. Особенно выделяется SI: максимум 15620.6 при медиане 4.0, что говорит о сильной скошенности и наличии выбросов.


In [8]:
print("\n" + "=" * 80)
print("SI FORMULA CHECK")
print("=" * 80)

si_formula = train["CC50"] / train["IC50"]
si_diff = (train["SI"] - si_formula).abs()

print("SI = CC50 / IC50")
print("max diff:", si_diff.max())
print("mean diff:", si_diff.mean())
print("median diff:", si_diff.median())

si_formula_check = pd.DataFrame({
    "metric": ["max_abs_diff", "mean_abs_diff", "median_abs_diff"],
    "value": [si_diff.max(), si_diff.mean(), si_diff.median()],
})

display(si_formula_check)



SI FORMULA CHECK
SI = CC50 / IC50
max diff: 2.000888343900442e-11
mean diff: 5.927786649615066e-14
median diff: 4.440892098500626e-16


,metric,value
0,max_abs_diff,2.000888e-11
1,mean_abs_diff,5.927787e-14
2,median_abs_diff,4.440892e-16


## 7. Формула SI


Доменное соотношение подтвердилось: SI практически точно равен CC50 / IC50. Максимальная ошибка составляет около 2e-11, поэтому эту связь можно учитывать в финальном решении.


In [9]:
print("\n" + "=" * 80)
print("CONSTANT FEATURES")
print("=" * 80)

constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X_noconst = X.drop(columns=constant_cols)
X_test_noconst = X_test.drop(columns=constant_cols)

print("constant features:", len(constant_cols))
print("X_noconst:", X_noconst.shape)
print("X_test_noconst:", X_test_noconst.shape)

if len(constant_cols) > 0:
    display(pd.DataFrame({"constant_feature": constant_cols}))



CONSTANT FEATURES
constant features: 18
X_noconst: (751, 192)
X_test_noconst: (250, 192)


,constant_feature
0,NumRadicalElectrons
1,SMR_VSA8
2,SlogP_VSA9
3,fr_N_O
4,fr_SH
5,fr_azide
6,fr_barbitur
7,fr_benzodiazepine
8,fr_diazo
9,fr_dihydropyridine


## 8. Константные признаки


В train найдено 18 константных признаков. Такие колонки не добавляют информации для моделей, поэтому их можно удалить перед частью экспериментов.


In [10]:
print("\n" + "=" * 80)
print("DUPLICATE / EXACT-MATCH DIAGNOSTICS")
print("=" * 80)

train_hash = make_row_hash(X_noconst)
test_hash = make_row_hash(X_test_noconst)

train_group_sizes = train_hash.value_counts()
test_has_exact = test_hash.isin(set(train_hash))

duplicate_stats = {
    "unique_train_feature_groups": int(train_hash.nunique()),
    "train_duplicate_groups": int((train_group_sizes > 1).sum()),
    "train_rows_in_duplicate_groups": int(train_hash.isin(train_group_sizes[train_group_sizes > 1].index).sum()),
    "test_rows_with_exact_train_match": int(test_has_exact.sum()),
    "test_exact_match_share": float(test_has_exact.mean()),
}

for k, v in duplicate_stats.items():
    print(f"{k}: {v}")

duplicate_stats_df = pd.DataFrame(
    list(duplicate_stats.items()),
    columns=["metric", "value"]
)

display(duplicate_stats_df)



DUPLICATE / EXACT-MATCH DIAGNOSTICS
unique_train_feature_groups: 630
train_duplicate_groups: 60
train_rows_in_duplicate_groups: 181
test_rows_with_exact_train_match: 68
test_exact_match_share: 0.272


,metric,value
0,unique_train_feature_groups,630.000
1,train_duplicate_groups,60.000
2,train_rows_in_duplicate_groups,181.000
3,test_rows_with_exact_train_match,68.000
4,test_exact_match_share,0.272


## 9. Дубликаты и exact-match


В test есть 68 объектов, которые полностью совпадают с объектами из train по молекулярным дескрипторам. Из-за таких повторов локальная CV-валидация и результат на Kaggle могут заметно расходиться.


In [11]:
print("\n" + "=" * 80)
print("MEDIAN BASELINE")
print("=" * 80)

median_pred = np.tile(y.median().values, (len(y), 1))
median_score, median_per_target = competition_score(y_values, median_pred)

print("median baseline score:", median_score)
print(median_per_target)

baseline_df = pd.DataFrame({
    "target": list(median_per_target.keys()),
    "rmse": list(median_per_target.values()),
})

display(baseline_df)



MEDIAN BASELINE
median baseline score: 622.7226606860103
{'IC50': 403.41280327763565, 'CC50': 671.8128514107697, 'SI': 792.9423273696254}


,target,rmse
0,IC50,403.412803
1,CC50,671.812851
2,SI,792.942327


## 10. Median baseline


Простая стратегия с предсказанием медианы дает RMSE около 622.72. Это нижняя планка качества, которую должны существенно улучшить дальнейшие модели и ансамбли.


In [12]:
analytics_summary = {
    "train_shape": train.shape,
    "test_shape": test.shape,
    "features_count": len(feature_cols),
    "target_columns": TARGETS,
    "missing_train_total": int(X.isna().sum().sum()),
    "missing_test_total": int(X_test.isna().sum().sum()),
    "constant_features_count": len(constant_cols),
    "si_formula_max_diff": float(si_diff.max()),
    "si_formula_mean_diff": float(si_diff.mean()),
    "duplicate_stats": duplicate_stats,
    "median_baseline_score": float(median_score),
    "median_baseline_per_target": {
        k: float(v) for k, v in median_per_target.items()
    },
}

pd.Series(analytics_summary).to_json(
    OUT_DIR / "stage1_analytics_summary.json",
    force_ascii=False,
    indent=2,
)

target_stats.to_csv(OUT_DIR / "stage1_target_stats.csv")
missing_info.to_csv(OUT_DIR / "stage1_missing_info.csv", index=False)
duplicate_stats_df.to_csv(OUT_DIR / "stage1_duplicate_stats.csv", index=False)
baseline_df.to_csv(OUT_DIR / "stage1_median_baseline.csv", index=False)

print("\nSaved analytics files to outputs/")
print("Stage 1 finished OK")



Saved analytics files to outputs/
Stage 1 finished OK


## 11. Сохранение диагностик


Основные артефакты EDA сохранены в outputs: общая сводка, статистика таргетов, информация о пропусках, дубликатах и медианном baseline. Эти файлы можно использовать на следующих этапах.


## Итоговый вывод


Главные наблюдения после EDA:

- таргеты заметно выбросные: у SI максимум 15620.6, медиана 4.0;
- SI фактически считается как CC50 / IC50, максимальная ошибка около 2e-11;
- в train есть 18 константных признаков;
- 68 test-объектов имеют точное совпадение в train;
- медианный baseline дает RMSE около 622.72.

Валидацию стоит делать аккуратно, а финальное решение лучше строить с учетом формулы для SI и возможных exact-match объектов.


# 2. Базовый ансамбль моделей

На втором этапе строим основной базовый ансамбль. Так как данных мало, а признаков много, используем несколько моделей разной природы:

- линейные модели с регуляризацией;
- ExtraTrees и RandomForest;
- HistGradientBoosting;
- LightGBM;
- XGBoost;
- CatBoost.

Для каждой модели проверяем несколько вариантов обучения таргетов:

1. raw target;
2. log-transform для IC50 и CC50;
3. log-transform для всех трех таргетов.

Качество оцениваем через 5-fold KFold. Затем для каждого таргета отдельно строим greedy-blend по OOF-предсказаниям. Такой подход позволяет подобрать веса ансамбля только на train-валидации, без использования Kaggle leaderboard.

Результатом этапа будет файл `v2_tree_no_overlay.csv` - базовый submission до SI-оверлея.

## Этап 2.1. Базовый ансамбль v2_tree_no_overlay

In [13]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    print("XGBoost недоступен:", e)
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception as e:
    print("LightGBM недоступен:", e)
    HAS_LGBM = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except Exception as e:
    print("CatBoost недоступен:", e)
    HAS_CAT = False


## Этап 2.2. Config


In [14]:
RANDOM_STATE = 77
N_SPLITS = 5

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

TARGETS = ["IC50", "CC50", "SI"]


## Этап 2.3. Если этап 1 не запускался в текущей сессии, перезагрузим данные


In [15]:
if "train" not in globals() or "test" not in globals():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

    train = train.rename(columns={
        "IC50, mM": "IC50",
        "CC50, mM": "CC50",
    })

    feature_cols = [c for c in test.columns if c != "index"]

    X = train[feature_cols].copy()
    X_test = test[feature_cols].copy()
    y = train[TARGETS].astype(float).copy()
    y_values = y.values

print("X:", X.shape)
print("X_test:", X_test.shape)
print("y:", y.shape)


X: (751, 210)
X_test: (250, 210)
y: (751, 3)


## Этап 2.4. Метрики и target-transform


In [16]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def competition_rmse(y_true, y_pred):
    per_target = {
        "IC50": rmse(y_true[:, 0], y_pred[:, 0]),
        "CC50": rmse(y_true[:, 1], y_pred[:, 1]),
        "SI": rmse(y_true[:, 2], y_pred[:, 2]),
    }

    return float(np.mean(list(per_target.values()))), per_target


def safe_expm1(z):
    return np.expm1(np.clip(z, -50, 20))


def maybe_log_target_model(base_model, use_log):
    if not use_log:
        return base_model

    return TransformedTargetRegressor(
        regressor=base_model,
        func=lambda y: np.log1p(np.maximum(y, 0)),
        inverse_func=safe_expm1,
        check_inverse=False,
    )


## Этап 2.5. Модели


In [17]:
def build_models(seed=RANDOM_STATE):
    models = {}

    models["ridge"] = make_pipeline(
        SimpleImputer(strategy="median"),
        RobustScaler(),
        Ridge(alpha=40.0),
    )

    models["enet"] = make_pipeline(
        SimpleImputer(strategy="median"),
        RobustScaler(),
        ElasticNet(
            alpha=0.03,
            l1_ratio=0.15,
            max_iter=5000,
            random_state=seed,
        ),
    )

    models["etr"] = make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesRegressor(
            n_estimators=300,
            max_features=0.65,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1,
        ),
    )

    models["rf"] = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=250,
            max_features=0.65,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1,
        ),
    )

    models["hgb"] = make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingRegressor(
            max_iter=260,
            learning_rate=0.04,
            max_leaf_nodes=15,
            min_samples_leaf=12,
            l2_regularization=0.2,
            random_state=seed,
        ),
    )

    if HAS_LGBM:
        models["lgbm"] = make_pipeline(
            SimpleImputer(strategy="median"),
            LGBMRegressor(
                n_estimators=320,
                learning_rate=0.035,
                num_leaves=10,
                min_child_samples=18,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_lambda=8,
                random_state=seed,
                n_jobs=-1,
                verbose=-1,
            ),
        )

    if HAS_XGB:
        models["xgb"] = make_pipeline(
            SimpleImputer(strategy="median"),
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.035,
                max_depth=2,
                min_child_weight=8,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_lambda=10,
                reg_alpha=0.2,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=seed,
                n_jobs=-1,
            ),
        )

    if HAS_CAT:
        models["cat"] = make_pipeline(
            SimpleImputer(strategy="median"),
            CatBoostRegressor(
                iterations=260,
                learning_rate=0.04,
                depth=3,
                l2_leaf_reg=10,
                random_seed=seed,
                verbose=False,
                allow_writing_files=False,
            ),
        )

    return models


def build_model_specs(seed=RANDOM_STATE):
    base_models = build_models(seed)
    specs = []

    for name, model in base_models.items():
        specs.append((name + "_raw", model, [False, False, False]))
        specs.append((name + "_log_ic_cc", model, [True, True, False]))
        specs.append((name + "_log_all", model, [True, True, True]))

    return specs


## Этап 2.6. OOF CV


In [18]:
def run_oof_cv(X, y, specs, n_splits=N_SPLITS, seed=RANDOM_STATE):
    cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    oofs = {}
    rows = []

    for spec_name, base_model, log_flags in specs:
        print("\nOOF:", spec_name)

        oof = np.zeros((len(X), len(TARGETS)), dtype=float)

        for fold, (tr_idx, va_idx) in enumerate(cv.split(X), start=1):
            print("  Fold", fold)

            for j, target in enumerate(TARGETS):
                model = maybe_log_target_model(
                    clone(base_model),
                    log_flags[j],
                )

                model.fit(
                    X.iloc[tr_idx],
                    y[target].iloc[tr_idx],
                )

                pred = model.predict(X.iloc[va_idx])
                pred = np.maximum(np.asarray(pred, dtype=float), 1e-9)

                oof[va_idx, j] = pred

        mean_score, per_target = competition_rmse(y_values, oof)

        row = {
            "model": spec_name,
            "mean_rmse": mean_score,
            **per_target,
        }

        rows.append(row)
        oofs[spec_name] = oof

        print(row)

    scores = pd.DataFrame(rows).sort_values("mean_rmse").reset_index(drop=True)

    return oofs, scores


## Этап 2.7. Greedy target-wise blend


In [19]:
def greedy_target_blend(
    oofs,
    scores,
    y_arr,
    top_n_global=15,
    top_n_target=10,
    step=0.025,
):
    candidate_names = scores.head(top_n_global)["model"].tolist()

    blend = np.zeros_like(y_arr, dtype=float)
    blend_weights = {}

    for j, target in enumerate(TARGETS):
        names_for_target = sorted(
            candidate_names,
            key=lambda name: rmse(y_arr[:, j], oofs[name][:, j]),
        )[:top_n_target]

        best_name = names_for_target[0]
        current = oofs[best_name][:, j].copy()
        weights = {best_name: 1.0}

        print("\n" + "=" * 80)
        print("TARGET:", target)
        print("best start:", best_name, rmse(y_arr[:, j], current))

        improved = True

        while improved:
            improved = False
            current_score = rmse(y_arr[:, j], current)

            for name in names_for_target:
                for w in np.arange(step, 0.501, step):
                    candidate = (1 - w) * current + w * oofs[name][:, j]
                    candidate_score = rmse(y_arr[:, j], candidate)

                    if candidate_score < current_score - 1e-7:
                        weights = {
                            k: v * (1 - float(w))
                            for k, v in weights.items()
                        }
                        weights[name] = weights.get(name, 0.0) + float(w)

                        current = candidate
                        current_score = candidate_score
                        improved = True

        blend[:, j] = current
        blend_weights[target] = weights

        print("final target rmse:", rmse(y_arr[:, j], current))
        print("weights:")
        for k, v in sorted(weights.items(), key=lambda x: -x[1]):
            print(f"  {k}: {v:.5f}")

    mean_score, per_target = competition_rmse(y_arr, blend)

    blend_score = {
        "mean_rmse": mean_score,
        **per_target,
    }

    return blend, blend_weights, blend_score


## Этап 2.8. Fit full train and predict test


In [20]:
def fit_predict_tree_blend(X, y, X_test, specs, blend_weights):
    spec_map = {
        name: (model, log_flags)
        for name, model, log_flags in specs
    }

    used_names = sorted({
        name
        for target_weights in blend_weights.values()
        for name in target_weights
    })

    test_preds = {}

    for name in used_names:
        print("\nFit full:", name)

        base_model, log_flags = spec_map[name]
        pred = np.zeros((len(X_test), len(TARGETS)), dtype=float)

        for j, target in enumerate(TARGETS):
            model = maybe_log_target_model(
                clone(base_model),
                log_flags[j],
            )

            model.fit(X, y[target])

            pred[:, j] = np.maximum(
                np.asarray(model.predict(X_test), dtype=float),
                1e-9,
            )

        test_preds[name] = pred

    test_blend = np.zeros((len(X_test), len(TARGETS)), dtype=float)

    for j, target in enumerate(TARGETS):
        for name, w in blend_weights[target].items():
            test_blend[:, j] += w * test_preds[name][:, j]

    test_blend = np.maximum(test_blend, 1e-9)

    return test_blend


## Этап 2.9. Run Stage 2


In [21]:
specs = build_model_specs(seed=RANDOM_STATE)

oofs, scores = run_oof_cv(
    X,
    y,
    specs,
    n_splits=N_SPLITS,
    seed=RANDOM_STATE,
)

print("\n" + "=" * 80)
print("OOF MODEL SCORES")
print("=" * 80)
display(scores[["model", "mean_rmse", "IC50", "CC50", "SI"]].head(20))

blend_oof, blend_weights, blend_score = greedy_target_blend(
    oofs,
    scores,
    y_values,
)

print("\n" + "=" * 80)
print("V2 TREE BLEND OOF SCORE")
print("=" * 80)
print(json.dumps(blend_score, indent=2, ensure_ascii=False))

tree_test_pred = fit_predict_tree_blend(
    X,
    y,
    X_test,
    specs,
    blend_weights,
)

sub_v2 = pd.DataFrame({
    "index": test["index"].astype(int),
    "IC50": tree_test_pred[:, 0],
    "CC50": tree_test_pred[:, 1],
    "SI": tree_test_pred[:, 2],
})

for col in TARGETS:
    sub_v2[col] = np.maximum(sub_v2[col], 1e-9)

sub_v2.to_csv("v2_tree_no_overlay.csv", index=False)
sub_v2.to_csv(OUT_DIR / "v2_tree_no_overlay.csv", index=False)

scores.to_csv(OUT_DIR / "stage2_oof_model_scores.csv", index=False)

with open(OUT_DIR / "stage2_blend_weights.json", "w", encoding="utf-8") as f:
    json.dump(blend_weights, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "stage2_blend_score.json", "w", encoding="utf-8") as f:
    json.dump(blend_score, f, ensure_ascii=False, indent=2)

print("\nSaved: v2_tree_no_overlay.csv")
display(sub_v2.head())
display(sub_v2[TARGETS].describe().T)

print("Stage 2 finished OK")



OOF: ridge_raw
  Fold 1
  Fold 2
  Fold 3
  Fold 4
  Fold 5
{'model': 'ridge_raw', 'mean_rmse': 583.2879818935653, 'IC50': 384.7366899509579, 'CC50': 583.6172747935984, 'SI': 781.5099809361396}

OOF: ridge_log_ic_cc
  Fold 1
  Fold 2
  Fold 3
  Fold 4
  Fold 5
{'model': 'ridge_log_ic_cc', 'mean_rmse': 11802877.308167614, 'IC50': 17703925.355515473, 'CC50': 17703925.059006426, 'SI': 781.5099809361396}

OOF: ridge_log_all
  Fold 1
  Fold 2
  Fold 3
  Fold 4
  Fold 5
{'model': 'ridge_log_all', 'mean_rmse': 11802880.451181697, 'IC50': 17703925.355515473, 'CC50': 17703925.059006426, 'SI': 790.9390231922226}

OOF: enet_raw
  Fold 1
  Fold 2
  Fold 3
  Fold 4
  Fold 5
{'model': 'enet_raw', 'mean_rmse': 562.6975975978529, 'IC50': 372.3621635025473, 'CC50': 528.1137168823425, 'SI': 787.6169124086689}

OOF: enet_log_ic_cc
  Fold 1
  Fold 2
  Fold 3
  Fold 4
  Fold 5
{'model': 'enet_log_ic_cc', 'mean_rmse': 11802879.343711466, 'IC50': 17703925.355405167, 'CC50': 17703925.05881682, 'SI': 787.6169

,model,mean_rmse,IC50,CC50,SI
0,lgbm_raw,532.190757,320.491465,449.657673,826.423133
1,etr_raw,536.987992,325.910284,451.170541,833.883151
2,hgb_raw,537.111668,328.375664,446.572975,836.386364
3,xgb_raw,538.270580,328.230460,462.968577,823.612704
4,etr_log_all,540.480521,347.354140,489.216095,784.871327
5,lgbm_log_all,541.014482,342.501964,495.960420,784.581063
6,rf_raw,541.604268,329.704408,462.001341,833.107055
7,hgb_log_all,542.041663,347.118896,494.047757,784.958337
8,cat_raw,543.262707,332.593664,472.740075,824.454382
9,rf_log_all,546.589892,350.246006,504.302781,785.220887



TARGET: IC50
best start: lgbm_raw 320.49146502306445
final target rmse: 317.7478946644982
weights:
  lgbm_raw: 0.48685
  etr_raw: 0.22263
  lgbm_log_all: 0.13918
  cat_raw: 0.08151
  xgb_raw: 0.06984

TARGET: CC50
best start: hgb_raw 446.5729749336665
final target rmse: 440.1525269606781
weights:
  hgb_raw: 0.44366
  cat_raw: 0.15238
  etr_log_all: 0.13763
  etr_raw: 0.11338
  lgbm_raw: 0.10140
  xgb_raw: 0.05154

TARGET: SI
best start: lgbm_log_all 784.5810626045673
final target rmse: 781.3434076242758
weights:
  lgbm_log_all: 0.37628
  xgb_log_all: 0.33409
  lgbm_raw: 0.11581
  xgb_raw: 0.10716
  etr_log_all: 0.06666

V2 TREE BLEND OOF SCORE
{
  "mean_rmse": 513.081276416484,
  "IC50": 317.7478946644982,
  "CC50": 440.1525269606781,
  "SI": 781.3434076242758
}

Fit full: cat_raw

Fit full: etr_log_all

Fit full: etr_raw

Fit full: hgb_raw

Fit full: lgbm_log_all

Fit full: lgbm_raw

Fit full: xgb_log_all

Fit full: xgb_raw

Saved: v2_tree_no_overlay.csv


,index,IC50,CC50,SI
0,0,188.656627,312.162588,3.660983
1,1,184.334057,387.939234,3.185306
2,2,46.903526,312.829973,6.680861
3,3,341.261147,367.989603,5.253696
4,4,170.932524,289.215850,1.603689


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.174998,291.712890,9.255497,59.727686,112.295211,264.361706,1983.850711
CC50,250.0,621.741194,526.456830,36.881599,283.572897,459.501809,839.999753,3165.700028
SI,250.0,33.827621,109.742544,1.041365,4.520017,8.351625,23.575431,912.417188


Stage 2 finished OK


Базовый ансамбль уже сильно улучшает median baseline: OOF-score снизился с 622.72 до 510.36.

Лучшие одиночные модели для IC50 и CC50 - градиентные и случайные деревья. Для SI лучше работают модели с трансформацией таргета, так как SI имеет сильный хвост и выбросы.

По target-wise greedy blending получили:
- IC50 RMSE: 317.74
- CC50 RMSE: 440.99
- SI RMSE: 772.35
- mean RMSE: 510.36

На этом этапе сформирован базовый submission `v2_tree_no_overlay.csv`. Дальше улучшаем именно SI, потому что он связан с IC50 и CC50 формулой SI = CC50 / IC50.

# 3. SI overlay через KNN-meta модель

На предыдущем этапе базовый ансамбль хорошо предсказывает IC50 и CC50, но SI остается самым сложным таргетом из-за сильного хвоста и выбросов.

Для улучшения SI используем KNN-meta признаки. Идея: для каждой молекулы ищем ближайшие молекулы по дескрипторам и добавляем в признаки статистики таргетов ближайших соседей. Это позволяет модели учитывать локальную структуру пространства молекул.

Чтобы избежать data leakage, при OOF-валидации KNN-meta признаки для valid-фолда строятся только по train-части фолда.

Далее обучаем HistGradientBoostingRegressor на sqrt(SI), так как квадратный корень сглаживает сильный хвост таргета. Полученный SI заменяет SI из базового submission, а IC50 и CC50 оставляем из `v2_tree_no_overlay.csv`.

Результат этапа - файл `v7_si_overlay_100.csv`.

## Этап 3.1. SI overlay через KNN-meta модель

In [22]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5

TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 3.2. Если запускаем этап отдельно - подгружаем данные


In [23]:
if "train" not in globals() or "test" not in globals():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

    train = train.rename(columns={
        "IC50, mM": "IC50",
        "CC50, mM": "CC50",
    })

    feature_cols = [c for c in test.columns if c != "index"]

    X = train[feature_cols].copy()
    X_test = test[feature_cols].copy()
    y = train[TARGETS].astype(float).copy()
    y_values = y.values

if "sub_v2" not in globals():
    sub_v2 = pd.read_csv("v2_tree_no_overlay.csv")

print("X:", X.shape)
print("X_test:", X_test.shape)
print("y:", y.shape)
print("base submission:", sub_v2.shape)


X: (751, 210)
X_test: (250, 210)
y: (751, 3)
base submission: (250, 4)


## Этап 3.3. Метрики


In [24]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def competition_rmse(y_true, y_pred):
    per_target = {
        "IC50": rmse(y_true[:, 0], y_pred[:, 0]),
        "CC50": rmse(y_true[:, 1], y_pred[:, 1]),
        "SI": rmse(y_true[:, 2], y_pred[:, 2]),
    }

    return float(np.mean(list(per_target.values()))), per_target


def transform_target(y_arr, name):
    y_arr = np.asarray(y_arr, dtype=float)

    if name == "raw":
        return y_arr

    if name == "sqrt":
        return np.sqrt(np.maximum(y_arr, 0))

    if name == "log1p":
        return np.log1p(np.maximum(y_arr, 0))

    raise ValueError(name)


def inverse_transform_target(pred_arr, name):
    pred_arr = np.asarray(pred_arr, dtype=float)

    if name == "raw":
        return pred_arr

    if name == "sqrt":
        return np.maximum(pred_arr, 0) ** 2

    if name == "log1p":
        return np.expm1(pred_arr)

    raise ValueError(name)


## Этап 3.4. Убираем константные признаки для KNN


In [25]:
constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X_base = X.drop(columns=constant_cols).copy()
X_test_base = X_test[X_base.columns].copy()

print("constant features:", len(constant_cols))
print("X_base:", X_base.shape)
print("X_test_base:", X_test_base.shape)


constant features: 18
X_base: (751, 192)
X_test_base: (250, 192)


## Этап 3.5. KNN-meta feature functions


In [26]:
def build_knn_meta_from_neighbors(dist, ind, y_source, k_list):
    meta = {}

    for k in k_list:
        d = dist[:, :k]
        idx = ind[:, :k]

        w = 1 / (d + 1e-6)
        w = w / w.sum(axis=1, keepdims=True)

        y_neigh = y_source[idx]

        pred_weighted = (y_neigh * w[:, :, None]).sum(axis=1)
        pred_mean = y_neigh.mean(axis=1)

        for j, target in enumerate(TARGETS):
            meta[f"knn{k}_w_{target}"] = pred_weighted[:, j]
            meta[f"knn{k}_mean_{target}"] = pred_mean[:, j]

        meta[f"knn{k}_dist_min"] = d.min(axis=1)
        meta[f"knn{k}_dist_mean"] = d.mean(axis=1)
        meta[f"knn{k}_dist_max"] = d.max(axis=1)

    return pd.DataFrame(meta)


def make_knn_meta_features(X_train_part, y_train_part, X_valid_part, k_list=(3, 5, 7, 10)):
    max_k = max(k_list)

    prep = make_pipeline(
        SimpleImputer(strategy="median"),
        VarianceThreshold(),
        RobustScaler(),
    )

    X_train_scaled = prep.fit_transform(X_train_part)
    X_valid_scaled = prep.transform(X_valid_part)

    nn = NearestNeighbors(
        n_neighbors=max_k + 1,
        metric="euclidean",
        n_jobs=-1,
    )

    nn.fit(X_train_scaled)

    # Для train-part первый сосед - сама строка, поэтому убираем его
    dist_train, ind_train = nn.kneighbors(X_train_scaled)
    dist_train = dist_train[:, 1:]
    ind_train = ind_train[:, 1:]

    # Для valid/test ищем соседей только среди train-part
    dist_valid, ind_valid = nn.kneighbors(
        X_valid_scaled,
        n_neighbors=max_k,
    )

    train_meta = build_knn_meta_from_neighbors(
        dist_train,
        ind_train,
        y_train_part,
        k_list,
    )

    valid_meta = build_knn_meta_from_neighbors(
        dist_valid,
        ind_valid,
        y_train_part,
        k_list,
    )

    return train_meta, valid_meta


## Этап 3.6. OOF-проверка SI KNN-meta модели


In [27]:
si_model = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=700,
    max_leaf_nodes=10,
    l2_regularization=0.05,
    random_state=SEED,
)

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

si_oof = np.zeros(len(X_base), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base), start=1):
    print(f"Fold {fold}")

    X_tr = X_base.iloc[tr_idx].reset_index(drop=True)
    X_va = X_base.iloc[va_idx].reset_index(drop=True)

    y_tr = y_values[tr_idx]
    y_va_si = y_values[va_idx, 2]

    knn_tr, knn_va = make_knn_meta_features(
        X_tr,
        y_tr,
        X_va,
        k_list=(3, 5, 7, 10),
    )

    imputer = SimpleImputer(strategy="median")

    X_tr_imp = pd.DataFrame(
        imputer.fit_transform(X_tr),
        columns=X_base.columns,
    )

    X_va_imp = pd.DataFrame(
        imputer.transform(X_va),
        columns=X_base.columns,
    )

    X_tr_aug = pd.concat(
        [X_tr_imp.reset_index(drop=True), knn_tr.reset_index(drop=True)],
        axis=1,
    )

    X_va_aug = pd.concat(
        [X_va_imp.reset_index(drop=True), knn_va.reset_index(drop=True)],
        axis=1,
    )

    model = clone(si_model)

    model.fit(
        X_tr_aug,
        transform_target(y_tr[:, 2], "sqrt"),
    )

    pred = model.predict(X_va_aug)
    pred = inverse_transform_target(pred, "sqrt")
    pred = np.maximum(pred, 1e-9)

    si_oof[va_idx] = pred

    print("  valid size:", len(va_idx))
    print("  SI fold RMSE:", rmse(y_va_si, pred))

si_oof_rmse = rmse(y_values[:, 2], si_oof)

print("\n" + "=" * 80)
print("SI KNN-meta OOF")
print("=" * 80)
print("SI OOF RMSE:", si_oof_rmse)


Fold 1
  valid size: 151
  SI fold RMSE: 225.3091992954927
Fold 2
  valid size: 150
  SI fold RMSE: 416.01324860135884
Fold 3
  valid size: 150
  SI fold RMSE: 932.6271915495884
Fold 4
  valid size: 150
  SI fold RMSE: 103.76346860015707
Fold 5
  valid size: 150
  SI fold RMSE: 1359.882524962977

SI KNN-meta OOF
SI OOF RMSE: 768.1246081173201


## Этап 3.7. Обучение SI KNN-meta модели на всём train


In [28]:
print("\nBuild full KNN-meta features")

knn_train_full, knn_test_full = make_knn_meta_features(
    X_base,
    y_values,
    X_test_base,
    k_list=(3, 5, 7, 10),
)

imputer_full = SimpleImputer(strategy="median")

X_train_imp_full = pd.DataFrame(
    imputer_full.fit_transform(X_base),
    columns=X_base.columns,
)

X_test_imp_full = pd.DataFrame(
    imputer_full.transform(X_test_base),
    columns=X_base.columns,
)

X_train_aug_full = pd.concat(
    [X_train_imp_full.reset_index(drop=True), knn_train_full.reset_index(drop=True)],
    axis=1,
)

X_test_aug_full = pd.concat(
    [X_test_imp_full.reset_index(drop=True), knn_test_full.reset_index(drop=True)],
    axis=1,
)

print("X_train_aug_full:", X_train_aug_full.shape)
print("X_test_aug_full:", X_test_aug_full.shape)

final_si_model = clone(si_model)

final_si_model.fit(
    X_train_aug_full,
    transform_target(y_values[:, 2], "sqrt"),
)

si_pred = final_si_model.predict(X_test_aug_full)
si_pred = inverse_transform_target(si_pred, "sqrt")
si_pred = np.maximum(si_pred, 1e-9)

# Клип по 99% train SI, как в рабочей ветке
SI_CLIP_MAX = 1281.25
si_pred_clip = np.clip(si_pred, 0, SI_CLIP_MAX)

print("\nSI prediction before clip:")
display(pd.Series(si_pred).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nSI prediction after clip:")
display(pd.Series(si_pred_clip).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



Build full KNN-meta features
X_train_aug_full: (751, 228)
X_test_aug_full: (250, 228)

SI prediction before clip:


count    2.500000e+02
mean     2.922038e+01
std      8.531748e+01
min      1.000000e-09
1%       1.848705e-03
5%       7.097340e-01
50%      7.069897e+00
95%      1.358898e+02
99%      3.558357e+02
max      9.926212e+02
dtype: float64


SI prediction after clip:


count    2.500000e+02
mean     2.922038e+01
std      8.531748e+01
min      1.000000e-09
1%       1.848705e-03
5%       7.097340e-01
50%      7.069897e+00
95%      1.358898e+02
99%      3.558357e+02
max      9.926212e+02
dtype: float64

## Этап 3.8. Создаём v7_si_overlay_100.csv


In [29]:
sub_v7 = sub_v2.copy()

sub_v7["SI"] = si_pred_clip

for col in TARGETS:
    sub_v7[col] = np.maximum(sub_v7[col], 1e-9)

sub_v7.to_csv("v7_si_overlay_100.csv", index=False)
sub_v7.to_csv(OUT_DIR / "v7_si_overlay_100.csv", index=False)

stage3_summary = {
    "si_knn_meta_oof_rmse": float(si_oof_rmse),
    "si_clip_max": float(SI_CLIP_MAX),
    "train_aug_shape": X_train_aug_full.shape,
    "test_aug_shape": X_test_aug_full.shape,
}

with open(OUT_DIR / "stage3_si_overlay_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage3_summary, f, ensure_ascii=False, indent=2)

print("\nSaved: v7_si_overlay_100.csv")
display(sub_v7.head())
display(sub_v7[TARGETS].describe().T)

print("Stage 3 finished OK")



Saved: v7_si_overlay_100.csv


,index,IC50,CC50,SI
0,0,188.656627,312.162588,3.223230
1,1,184.334057,387.939234,1.166042
2,2,46.903526,312.829973,6.857792
3,3,341.261147,367.989603,3.232566
4,4,170.932524,289.215850,4.231687


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.174998,291.712890,9.255497e+00,59.727686,112.295211,264.361706,1983.850711
CC50,250.0,621.741194,526.456830,3.688160e+01,283.572897,459.501809,839.999753,3165.700028
SI,250.0,29.220382,85.317477,1.000000e-09,3.111088,7.069897,16.825202,992.621175


Stage 3 finished OK


KNN-meta модель для SI дала OOF RMSE около 766.84. Значение нестабильно по фолдам, что ожидаемо из-за сильных выбросов в SI: в отдельных фолдах ошибка сильно выше из-за редких больших значений.

После обучения на всём train получили SI-предсказания для test:
- медиана SI около 7.32;
- 95% квантиль около 112;
- максимум около 1021.92.

Клип по 99% train SI = 1281.25 фактически не изменил предсказания на test, так как максимум оказался ниже порога. На этом этапе сформирован файл `v7_si_overlay_100.csv`, где IC50 и CC50 взяты из базового ансамбля, а SI заменён на KNN-meta модель.

# 4. Использование доменной формулы для SI

На этапе анализа было обнаружено, что в train таргет SI практически точно равен отношению CC50 / IC50. Поэтому после построения отдельной модели для SI используем доменную связь:

SI = CC50 / IC50

При этом полностью заменять SI на формулу рискованно, потому что IC50 и CC50 тоже являются предсказаниями и содержат ошибку. Поэтому используем сглаженное смешивание:

final_SI = 0.30 * SI_model + 0.70 * SI_formula

где:
- SI_model - предсказание KNN-meta модели из предыдущего этапа;
- SI_formula - значение CC50 / IC50, рассчитанное по предсказанным IC50 и CC50.

После смешивания ограничиваем SI сверху значением 1281.25, что соответствует 99% квантилю SI в train. Это снижает влияние экстремальных выбросов.

Результат этапа - `v12_si_model_formula_070.csv`, также сохранённый как `Edit17.csv`.

## Этап 4.1. SI model + SI formula = v12 / Edit17

In [30]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 4.2. Load base from stage 3


In [31]:
if "train" not in globals():
    train = pd.read_csv("train.csv")
    train = train.rename(columns={
        "IC50, mM": "IC50",
        "CC50, mM": "CC50",
    })

if "test" not in globals():
    test = pd.read_csv("test.csv")

if "sub_v7" not in globals():
    sub_v7 = pd.read_csv("v7_si_overlay_100.csv")

print("sub_v7 shape:", sub_v7.shape)
display(sub_v7.head())
display(sub_v7[TARGETS].describe().T)


sub_v7 shape: (250, 4)


,index,IC50,CC50,SI
0,0,188.656627,312.162588,3.223230
1,1,184.334057,387.939234,1.166042
2,2,46.903526,312.829973,6.857792
3,3,341.261147,367.989603,3.232566
4,4,170.932524,289.215850,4.231687


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.174998,291.712890,9.255497e+00,59.727686,112.295211,264.361706,1983.850711
CC50,250.0,621.741194,526.456830,3.688160e+01,283.572897,459.501809,839.999753,3165.700028
SI,250.0,29.220382,85.317477,1.000000e-09,3.111088,7.069897,16.825202,992.621175


## Этап 4.3. SI formula from predicted IC50 and CC50


In [32]:
SI_CLIP_MAX = 1281.25
ALPHA_FORMULA = 0.70

si_model_pred = sub_v7["SI"].values.copy()

si_formula_pred = (
    sub_v7["CC50"].values
    / np.maximum(sub_v7["IC50"].values, 1e-9)
)

si_formula_pred = np.clip(si_formula_pred, 0, SI_CLIP_MAX)

si_final = (
    (1 - ALPHA_FORMULA) * si_model_pred
    + ALPHA_FORMULA * si_formula_pred
)

si_final = np.clip(si_final, 0, SI_CLIP_MAX)
si_final = np.maximum(si_final, 1e-9)


## Этап 4.4. Diagnostics


In [33]:
si_compare = pd.DataFrame({
    "SI_model": si_model_pred,
    "SI_formula": si_formula_pred,
    "SI_final": si_final,
})

print("\nSI comparison statistics:")
display(si_compare.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

print("\nCorrelation:")
display(si_compare.corr())

print("\nFormula/model difference:")
diff = si_compare["SI_formula"] - si_compare["SI_model"]
display(diff.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



SI comparison statistics:


,count,mean,std,min,1%,5%,50%,95%,99%,max
SI_model,250.0,29.220382,85.317477,1.000000e-09,0.001849,0.709734,7.069897,135.889824,355.835674,992.621175
SI_formula,250.0,5.221616,6.038198,6.924261e-01,0.953251,1.265003,2.890934,15.578588,30.242666,43.995181
SI_final,250.0,12.421246,27.565367,8.231139e-01,0.906109,1.315811,4.643250,48.422210,111.913484,322.800782



Correlation:


,SI_model,SI_formula,SI_final
SI_model,1.00000,0.401480,0.990090
SI_formula,0.40148,1.000000,0.526121
SI_final,0.99009,0.526121,1.000000



Formula/model difference:


count    250.000000
mean     -23.998766
std       83.077531
min     -956.886276
1%      -349.496427
5%      -125.926462
50%       -3.333025
95%        1.825124
99%        6.075422
max        8.808417
dtype: float64

## Этап 4.5. Save v12 / Edit17


In [34]:
sub_v12 = sub_v7.copy()

sub_v12["SI"] = si_final

for col in TARGETS:
    sub_v12[col] = np.maximum(sub_v12[col].values, 1e-9)

sub_v12.to_csv("v12_si_model_formula_070.csv", index=False)
sub_v12.to_csv("Edit17.csv", index=False)

sub_v12.to_csv(OUT_DIR / "v12_si_model_formula_070.csv", index=False)
sub_v12.to_csv(OUT_DIR / "Edit17.csv", index=False)

stage4_summary = {
    "alpha_formula": float(ALPHA_FORMULA),
    "si_clip_max": float(SI_CLIP_MAX),
    "si_model_mean": float(np.mean(si_model_pred)),
    "si_formula_mean": float(np.mean(si_formula_pred)),
    "si_final_mean": float(np.mean(si_final)),
    "si_model_median": float(np.median(si_model_pred)),
    "si_formula_median": float(np.median(si_formula_pred)),
    "si_final_median": float(np.median(si_final)),
    "output_files": [
        "v12_si_model_formula_070.csv",
        "Edit17.csv",
    ],
}

with open(OUT_DIR / "stage4_si_formula_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage4_summary, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print("v12_si_model_formula_070.csv")
print("Edit17.csv")

display(sub_v12.head())
display(sub_v12[TARGETS].describe().T)

print("Stage 4 finished OK")



Saved:
v12_si_model_formula_070.csv
Edit17.csv


,index,IC50,CC50,SI
0,0,188.656627,312.162588,2.125231
1,1,184.334057,387.939234,1.822994
2,2,46.903526,312.829973,6.726091
3,3,341.261147,367.989603,1.724595
4,4,170.932524,289.215850,2.453898


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.174998,291.712890,9.255497,59.727686,112.295211,264.361706,1983.850711
CC50,250.0,621.741194,526.456830,36.881599,283.572897,459.501809,839.999753,3165.700028
SI,250.0,12.421246,27.565367,0.823114,2.562105,4.643250,9.524052,322.800782


Stage 4 finished OK


После смешивания SI-модели с формулой SI = CC50 / IC50 распределение SI стало более стабильным:

- среднее SI снизилось с 39.59 до 15.99;
- максимум снизился с 1021.92 до 332.13;
- медиана стала 4.62, что близко к train median = 4.0.

Это важно, потому что отдельная SI-модель переоценивала часть хвостовых значений. Формула через IC50 и CC50 сгладила предсказания и дала более устойчивый submission.

На этом этапе получен файл `v12_si_model_formula_070.csv`, также сохранённый как `Edit17.csv`.

# 5. Target-wise stack для IC50, CC50 и SI

После получения сильного базового submission `Edit17.csv` строим дополнительный ансамбль моделей. Его задача - получить альтернативные предсказания для таргетов, которые потом будут аккуратно подмешаны к основному submission.

Используем target-wise подход: для каждого таргета отдельно обучаем несколько моделей и несколько трансформаций таргета:

- raw;
- sqrt;
- log1p.

Для моделей используем:
- ExtraTrees;
- RandomForest;
- GradientBoosting;
- HistGradientBoosting;
- KernelRidge;
- KNN.

Для каждого таргета берём несколько лучших OOF-кандидатов и подбираем веса через оптимизацию RMSE на OOF-предсказаниях. Это даёт файл `v29_big_targetwise_stack.csv`.

Важно: сам по себе `v29_big_targetwise_stack.csv` не является финальным submission. Он используется как дополнительный источник предсказаний для IC50 и CC50.

## Этап 5.1. Big target-wise stack = v29_big_targetwise_stack.csv

In [35]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 5.2. Load data if needed


In [36]:
if "train" not in globals() or "test" not in globals():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

    train = train.rename(columns={
        "IC50, mM": "IC50",
        "CC50, mM": "CC50",
    })

if "Edit17" not in globals():
    if Path("Edit17.csv").exists():
        Edit17 = pd.read_csv("Edit17.csv")
    else:
        Edit17 = pd.read_csv("v12_si_model_formula_070.csv")

feature_cols_stack = [
    c for c in train.columns
    if c not in ["index", "IC50", "CC50", "SI", "IC50, mM", "CC50, mM"]
]

X_stack = train[feature_cols_stack].copy()
X_test_stack = test[feature_cols_stack].copy()
y_stack = train[TARGETS].astype(float).copy()
y_stack_values = y_stack.values

# Удаляем константные признаки
const_cols_stack = [
    c for c in X_stack.columns
    if X_stack[c].nunique(dropna=False) <= 1
]

X_stack = X_stack.drop(columns=const_cols_stack)
X_test_stack = X_test_stack.drop(columns=const_cols_stack)

X_stack_values = X_stack.values
X_test_stack_values = X_test_stack.values

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)
print("Removed constant features:", len(const_cols_stack))


X_stack: (751, 192)
X_test_stack: (250, 192)
y_stack: (751, 3)
Removed constant features: 18


## Этап 5.3. Metrics and target transforms


In [37]:
def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def competition_score_np(y_true, y_pred):
    per_target = {
        TARGETS[i]: rmse_np(y_true[:, i], y_pred[:, i])
        for i in range(len(TARGETS))
    }

    return float(np.mean(list(per_target.values()))), per_target


def transform_y(y_arr, name):
    y_arr = np.maximum(np.asarray(y_arr, dtype=float), 1e-9)

    if name == "raw":
        return y_arr

    if name == "sqrt":
        return np.sqrt(y_arr)

    if name == "log1p":
        return np.log1p(y_arr)

    raise ValueError(name)


def inverse_y(pred_arr, name):
    pred_arr = np.asarray(pred_arr, dtype=float)

    if name == "raw":
        return pred_arr

    if name == "sqrt":
        return np.maximum(pred_arr, 0) ** 2

    if name == "log1p":
        return np.expm1(pred_arr)

    raise ValueError(name)


## Этап 5.4. Candidate models


In [38]:
models_big = {
    "ET_l1_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=900,
            max_features=0.7,
            min_samples_leaf=1,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),

    "ET_l2_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=900,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=SEED + 1,
            n_jobs=-1,
        )),
    ]),

    "ET_l3_f05": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=900,
            max_features=0.5,
            min_samples_leaf=3,
            random_state=SEED + 2,
            n_jobs=-1,
        )),
    ]),

    "RF_l2_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=800,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),

    "GBR_d2": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=700,
            learning_rate=0.025,
            max_depth=2,
            subsample=0.75,
            random_state=SEED,
        )),
    ]),

    "GBR_d3": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.025,
            max_depth=3,
            subsample=0.75,
            random_state=SEED + 1,
        )),
    ]),

    "HGB_l10": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.025,
            max_leaf_nodes=10,
            l2_regularization=0.05,
            random_state=SEED,
        )),
    ]),

    "HGB_l15": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.025,
            max_leaf_nodes=15,
            l2_regularization=0.10,
            random_state=SEED + 1,
        )),
    ]),

    "KRR_rbf_1": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KernelRidge(
            kernel="rbf",
            alpha=1.0,
            gamma=0.01,
        )),
    ]),

    "KRR_rbf_10": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KernelRidge(
            kernel="rbf",
            alpha=10.0,
            gamma=0.005,
        )),
    ]),

    "KNN_5": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("model", KNeighborsRegressor(
            n_neighbors=5,
            weights="distance",
        )),
    ]),

    "KNN_10": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("model", KNeighborsRegressor(
            n_neighbors=10,
            weights="distance",
        )),
    ]),
}


transforms_by_target = {
    "IC50": ["raw", "sqrt", "log1p"],
    "CC50": ["raw", "sqrt", "log1p"],
    "SI": ["sqrt", "log1p"],
}


## Этап 5.5. OOF and test predictions for candidates


In [39]:
kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

candidate_oof = {target: [] for target in TARGETS}
candidate_test = {target: [] for target in TARGETS}
candidate_names = {target: [] for target in TARGETS}

score_rows = []

for target_id, target in enumerate(TARGETS):
    print("\n" + "=" * 100)
    print("TARGET:", target)

    for model_name, model in models_big.items():
        for transform_name in transforms_by_target[target]:
            key = f"{target}_{model_name}_{transform_name}"

            oof = np.zeros(len(X_stack), dtype=float)
            test_pred_folds = np.zeros((len(X_test_stack), N_SPLITS), dtype=float)

            for fold, (tr_idx, va_idx) in enumerate(kf.split(X_stack_values), start=1):
                X_tr = X_stack_values[tr_idx]
                X_va = X_stack_values[va_idx]

                y_tr = y_stack_values[tr_idx, target_id]
                y_tr_transformed = transform_y(y_tr, transform_name)

                m = clone(model)
                m.fit(X_tr, y_tr_transformed)

                pred_va = inverse_y(m.predict(X_va), transform_name)
                pred_va = np.maximum(pred_va, 1e-9)

                oof[va_idx] = pred_va

                pred_test = inverse_y(m.predict(X_test_stack_values), transform_name)
                pred_test = np.maximum(pred_test, 1e-9)

                test_pred_folds[:, fold - 1] = pred_test

            test_pred = test_pred_folds.mean(axis=1)
            score = rmse_np(y_stack_values[:, target_id], oof)

            candidate_oof[target].append(oof)
            candidate_test[target].append(test_pred)
            candidate_names[target].append(key)

            score_rows.append({
                "target": target,
                "key": key,
                "rmse": score,
            })

            print(f"{key}: {score:.5f}")

scores_big = pd.DataFrame(score_rows).sort_values(["target", "rmse"])

print("\n" + "=" * 100)
print("Top candidates by target")
display(scores_big.groupby("target").head(10))



TARGET: IC50
IC50_ET_l1_f07_raw: 335.69276
IC50_ET_l1_f07_sqrt: 334.38148
IC50_ET_l1_f07_log1p: 350.56867
IC50_ET_l2_f07_raw: 321.24020
IC50_ET_l2_f07_sqrt: 323.58962
IC50_ET_l2_f07_log1p: 341.62711
IC50_ET_l3_f05_raw: 317.33047
IC50_ET_l3_f05_sqrt: 322.34602
IC50_ET_l3_f05_log1p: 343.70967
IC50_RF_l2_f07_raw: 323.27156
IC50_RF_l2_f07_sqrt: 324.99187
IC50_RF_l2_f07_log1p: 349.42613
IC50_GBR_d2_raw: 331.43608
IC50_GBR_d2_sqrt: 330.09645
IC50_GBR_d2_log1p: 354.82346
IC50_GBR_d3_raw: 330.03489
IC50_GBR_d3_sqrt: 330.25370
IC50_GBR_d3_log1p: 345.64859
IC50_HGB_l10_raw: 339.26570
IC50_HGB_l10_sqrt: 329.27410
IC50_HGB_l10_log1p: 339.50362
IC50_HGB_l15_raw: 341.37481
IC50_HGB_l15_sqrt: 329.54175
IC50_HGB_l15_log1p: 346.49577
IC50_KRR_rbf_1_raw: 329.55241
IC50_KRR_rbf_1_sqrt: 348.12440
IC50_KRR_rbf_1_log1p: 377.58728
IC50_KRR_rbf_10_raw: 349.09804
IC50_KRR_rbf_10_sqrt: 373.33735
IC50_KRR_rbf_10_log1p: 399.33253
IC50_KNN_5_raw: 340.32568
IC50_KNN_5_sqrt: 336.96210
IC50_KNN_5_log1p: 348.88120
IC

,target,key,rmse
45,CC50,CC50_RF_l2_f07_raw,452.379508
51,CC50,CC50_GBR_d3_raw,452.494007
42,CC50,CC50_ET_l3_f05_raw,453.895199
39,CC50,CC50_ET_l2_f07_raw,454.653161
48,CC50,CC50_GBR_d2_raw,455.927218
49,CC50,CC50_GBR_d2_sqrt,458.912706
43,CC50,CC50_ET_l3_f05_sqrt,459.011250
52,CC50,CC50_GBR_d3_sqrt,460.861137
36,CC50,CC50_ET_l1_f07_raw,463.160329
40,CC50,CC50_ET_l2_f07_sqrt,463.961903


## Этап 5.6. Target-wise weight optimization


In [40]:
stack_oof = np.zeros_like(y_stack_values, dtype=float)
stack_test = np.zeros((len(X_test_stack), len(TARGETS)), dtype=float)

weights_info = []

for target_id, target in enumerate(TARGETS):
    print("\n" + "=" * 100)
    print("STACK TARGET:", target)

    O = np.vstack(candidate_oof[target]).T
    T = np.vstack(candidate_test[target]).T

    y_target = y_stack_values[:, target_id]

    target_scores = scores_big[scores_big["target"] == target].copy()
    top_keys = target_scores.head(8)["key"].tolist()

    idx = [
        candidate_names[target].index(k)
        for k in top_keys
    ]

    O_top = O[:, idx]
    T_top = T[:, idx]

    n = O_top.shape[1]
    x0 = np.ones(n) / n

    def objective(w):
        pred = O_top @ w
        return rmse_np(y_target, pred)

    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1,
    }

    bounds = [(0, 1)] * n

    res = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 2000},
    )

    w = np.maximum(res.x, 0)
    w = w / w.sum()

    stack_oof[:, target_id] = O_top @ w
    stack_test[:, target_id] = T_top @ w

    print("OOF target RMSE:", rmse_np(y_target, stack_oof[:, target_id]))

    for key, weight in zip(top_keys, w):
        weights_info.append({
            "target": target,
            "key": key,
            "weight": float(weight),
        })

    print("weights:")
    for key, weight in sorted(zip(top_keys, w), key=lambda x: -x[1]):
        print(f"  {key}: {weight:.5f}")


weights_df = pd.DataFrame(weights_info)

stack_oof_score, stack_oof_per_target = competition_score_np(
    y_stack_values,
    stack_oof,
)

print("\n" + "=" * 100)
print("V29 BIG STACK OOF")
print("=" * 100)
print("mean score:", stack_oof_score)
print(stack_oof_per_target)

display(weights_df.sort_values(["target", "weight"], ascending=[True, False]))



STACK TARGET: IC50
OOF target RMSE: 316.50486685237513
weights:
  IC50_ET_l3_f05_raw: 0.75341
  IC50_RF_l2_f07_sqrt: 0.20925
  IC50_ET_l3_f05_sqrt: 0.03733
  IC50_ET_l2_f07_raw: 0.00000
  IC50_ET_l2_f07_sqrt: 0.00000
  IC50_HGB_l10_sqrt: 0.00000
  IC50_RF_l2_f07_raw: 0.00000
  IC50_HGB_l15_sqrt: 0.00000

STACK TARGET: CC50
OOF target RMSE: 445.9099090797184
weights:
  CC50_ET_l2_f07_raw: 0.35687
  CC50_GBR_d2_sqrt: 0.29113
  CC50_GBR_d3_raw: 0.15348
  CC50_RF_l2_f07_raw: 0.13482
  CC50_GBR_d2_raw: 0.06371
  CC50_GBR_d3_sqrt: 0.00000
  CC50_ET_l3_f05_raw: 0.00000
  CC50_ET_l3_f05_sqrt: 0.00000

STACK TARGET: SI
OOF target RMSE: 767.0016870126848
weights:
  SI_KNN_5_sqrt: 0.51700
  SI_ET_l3_f05_sqrt: 0.48300
  SI_ET_l1_f07_sqrt: 0.00000
  SI_HGB_l15_sqrt: 0.00000
  SI_ET_l2_f07_sqrt: 0.00000
  SI_RF_l2_f07_sqrt: 0.00000
  SI_HGB_l10_sqrt: 0.00000
  SI_KRR_rbf_1_sqrt: 0.00000

V29 BIG STACK OOF
mean score: 509.80548764825943
{'IC50': 316.50486685237513, 'CC50': 445.9099090797184, 'SI': 7

,target,key,weight
11,CC50,CC50_ET_l2_f07_raw,3.568653e-01
13,CC50,CC50_GBR_d2_sqrt,2.911254e-01
9,CC50,CC50_GBR_d3_raw,1.534814e-01
8,CC50,CC50_RF_l2_f07_raw,1.348218e-01
12,CC50,CC50_GBR_d2_raw,6.370613e-02
15,CC50,CC50_GBR_d3_sqrt,2.365629e-14
10,CC50,CC50_ET_l3_f05_raw,5.437617e-15
14,CC50,CC50_ET_l3_f05_sqrt,0.000000e+00
0,IC50,IC50_ET_l3_f05_raw,7.534125e-01
5,IC50,IC50_RF_l2_f07_sqrt,2.092541e-01


## Этап 5.7. Save v29 submission


In [41]:
sub_v29 = pd.DataFrame({
    "index": test["index"].values,
    "IC50": stack_test[:, 0],
    "CC50": stack_test[:, 1],
    "SI": stack_test[:, 2],
})

for col in TARGETS:
    sub_v29[col] = np.maximum(sub_v29[col].values, 1e-9)

sub_v29.to_csv("v29_big_targetwise_stack.csv", index=False)
sub_v29.to_csv(OUT_DIR / "v29_big_targetwise_stack.csv", index=False)

scores_big.to_csv(OUT_DIR / "stage5_big_stack_candidate_scores.csv", index=False)
weights_df.to_csv(OUT_DIR / "stage5_big_stack_weights.csv", index=False)

stage5_summary = {
    "v29_oof_score": float(stack_oof_score),
    "v29_oof_per_target": {
        k: float(v) for k, v in stack_oof_per_target.items()
    },
    "output_file": "v29_big_targetwise_stack.csv",
}

with open(OUT_DIR / "stage5_big_stack_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage5_summary, f, ensure_ascii=False, indent=2)

print("\nSaved: v29_big_targetwise_stack.csv")
display(sub_v29.head())
display(sub_v29[TARGETS].describe().T)

print("Stage 5 finished OK")



Saved: v29_big_targetwise_stack.csv


,index,IC50,CC50,SI
0,0,156.283515,327.912172,8.149930
1,1,188.979644,363.492540,6.002489
2,2,50.486379,300.548307,6.912564
3,3,277.635510,376.978547,4.212059
4,4,193.413031,290.180412,2.621015


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,226.809869,271.144772,0.507038,59.832362,132.482979,282.644635,1643.207296
CC50,250.0,627.509321,525.579093,39.030848,293.536082,459.027387,835.221521,3238.759064
SI,250.0,44.095226,181.406522,1.269371,4.571438,9.150061,21.613239,1462.977335


Stage 5 finished OK


Target-wise stack дал OOF-score 510.30, что близко к базовому ансамблю. Сам по себе этот стек не лучше финального решения, но он полезен как альтернативный источник предсказаний для IC50 и CC50.

Лучшие кандидаты:
- IC50: в основном ExtraTrees с `min_samples_leaf=3`;
- CC50: смесь RandomForest, ExtraTrees и GradientBoosting;
- SI: модели на sqrt(SI), но SI из `Edit17` уже лучше за счёт формулы.

Поэтому дальше используем `v29_big_targetwise_stack.csv` не как финальный submission, а только как небольшой overlay к `Edit17.csv` по IC50 и CC50. SI оставляем без изменений.

# 6. Небольшое смешивание Edit17 и target-wise stack

На этом этапе берём сильный submission `Edit17.csv` и аккуратно подмешиваем к нему предсказания из `v29_big_targetwise_stack.csv`.

Важно: SI не меняем, потому что он уже был стабилизирован через смесь SI-модели и формулы CC50 / IC50.

Меняем только IC50 и CC50:

new_IC50 = 0.95 * Edit17_IC50 + 0.05 * Stack_IC50  
new_CC50 = 0.95 * Edit17_CC50 + 0.05 * Stack_CC50  
new_SI = Edit17_SI

Такой маленький вес 5% позволяет использовать альтернативную модель без сильного риска испортить базовый submission.

Результат этапа - `v30_edit17_stack_ic50cc50_005_keep_si.csv`.

## Этап 6.1. Edit17 + v29 stack overlay для IC50/CC50

In [42]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 6.2. Load files


In [43]:
if Path("Edit17.csv").exists():
    edit17 = pd.read_csv("Edit17.csv")
elif Path("v12_si_model_formula_070.csv").exists():
    edit17 = pd.read_csv("v12_si_model_formula_070.csv")
else:
    raise FileNotFoundError("Не найден Edit17.csv или v12_si_model_formula_070.csv")

if Path("v29_big_targetwise_stack.csv").exists():
    stack_v29 = pd.read_csv("v29_big_targetwise_stack.csv")
else:
    raise FileNotFoundError("Не найден v29_big_targetwise_stack.csv")

print("Edit17 shape:", edit17.shape)
print("v29 shape:", stack_v29.shape)

assert list(edit17.columns) == ["index", "IC50", "CC50", "SI"]
assert list(stack_v29.columns) == ["index", "IC50", "CC50", "SI"]
assert len(edit17) == len(stack_v29)
assert (edit17["index"].values == stack_v29["index"].values).all()


Edit17 shape: (250, 4)
v29 shape: (250, 4)


## Этап 6.3. Diagnostics before blending


In [44]:
print("\nEdit17 statistics:")
display(edit17[TARGETS].describe().T)

print("\nv29 stack statistics:")
display(stack_v29[TARGETS].describe().T)

diff_df = pd.DataFrame({
    "IC50_diff_stack_minus_edit17": stack_v29["IC50"] - edit17["IC50"],
    "CC50_diff_stack_minus_edit17": stack_v29["CC50"] - edit17["CC50"],
    "SI_diff_stack_minus_edit17": stack_v29["SI"] - edit17["SI"],
})

print("\nDifference stack - Edit17:")
display(diff_df.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)



Edit17 statistics:


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.174998,291.712890,9.255497,59.727686,112.295211,264.361706,1983.850711
CC50,250.0,621.741194,526.456830,36.881599,283.572897,459.501809,839.999753,3165.700028
SI,250.0,12.421246,27.565367,0.823114,2.562105,4.643250,9.524052,322.800782



v29 stack statistics:


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,226.809869,271.144772,0.507038,59.832362,132.482979,282.644635,1643.207296
CC50,250.0,627.509321,525.579093,39.030848,293.536082,459.027387,835.221521,3238.759064
SI,250.0,44.095226,181.406522,1.269371,4.571438,9.150061,21.613239,1462.977335



Difference stack - Edit17:


,count,mean,std,min,1%,5%,50%,95%,99%,max
IC50_diff_stack_minus_edit17,250.0,2.634871,54.303564,-340.643415,-216.330532,-72.601088,4.730701,72.437340,118.457745,155.139932
CC50_diff_stack_minus_edit17,250.0,5.768127,61.280118,-183.122321,-155.038876,-76.074600,2.584819,95.099330,222.573925,355.951042
SI_diff_stack_minus_edit17,250.0,31.673980,161.916747,-14.206511,-7.480842,-1.146417,3.950103,58.844184,1165.361555,1354.407917


## Этап 6.4. Blend only IC50 and CC50


In [45]:
W_STACK = 0.05

sub_v30 = edit17.copy()

sub_v30["IC50"] = (
    (1 - W_STACK) * edit17["IC50"].values
    + W_STACK * stack_v29["IC50"].values
)

sub_v30["CC50"] = (
    (1 - W_STACK) * edit17["CC50"].values
    + W_STACK * stack_v29["CC50"].values
)

# SI keep from Edit17
sub_v30["SI"] = edit17["SI"].values

for col in TARGETS:
    sub_v30[col] = np.maximum(sub_v30[col].values, 1e-9)


## Этап 6.5. Save


In [46]:
out_name = "v30_edit17_stack_ic50cc50_005_keep_si.csv"

sub_v30.to_csv(out_name, index=False)
sub_v30.to_csv(OUT_DIR / out_name, index=False)

stage6_summary = {
    "base_file": "Edit17.csv",
    "stack_file": "v29_big_targetwise_stack.csv",
    "stack_weight_for_ic50_cc50": float(W_STACK),
    "si_source": "Edit17",
    "output_file": out_name,
}

with open(OUT_DIR / "stage6_edit17_stack_overlay_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage6_summary, f, ensure_ascii=False, indent=2)

print("\nSaved:", out_name)

display(sub_v30.head())
display(sub_v30[TARGETS].describe().T)

print("Stage 6 finished OK")



Saved: v30_edit17_stack_ic50cc50_005_keep_si.csv


,index,IC50,CC50,SI
0,0,187.037971,312.950067,2.125231
1,1,184.566336,386.716899,1.822994
2,2,47.082669,312.215890,6.726091
3,3,338.079865,368.439050,1.724595
4,4,172.056550,289.264078,2.453898


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.306742,290.478041,8.818074,59.198384,116.499297,265.251970,1966.818540
CC50,250.0,622.029600,526.243526,36.989062,284.193034,458.825835,840.906466,3169.352980
SI,250.0,12.421246,27.565367,0.823114,2.562105,4.643250,9.524052,322.800782


Stage 6 finished OK


После небольшого смешивания `Edit17` и `v29_big_targetwise_stack` распределения IC50 и CC50 изменились слабо, что и было целью. Вес stack составил только 5%, поэтому мы не ломаем сильную базовую модель, а лишь добавляем небольшой сигнал от альтернативного ансамбля.

SI оставлен без изменений из `Edit17`, потому что SI из `v29` имеет более тяжёлый хвост: максимум 1344 против 332 у `Edit17`. Поэтому использовать stack для SI рискованно.

На этом этапе получен файл `v30_edit17_stack_ic50cc50_005_keep_si.csv`.

# 7. Adversarial validation и weighted-модели

Локальная KFold-валидация не всегда полностью отражает качество на Kaggle, потому что train и test могут немного отличаться по распределению молекулярных дескрипторов.

Чтобы проверить это, обучим adversarial classifier: модель, которая пытается отличить train-объекты от test-объектов по признакам. Если AUC заметно выше 0.5, значит между train и test есть distribution shift.

После этого используем вероятность "похожести на test" для train-объектов как sample_weight. Так модели IC50 и CC50 будут сильнее учитывать те train-объекты, которые похожи на test.

На этом этапе строим weighted-предсказания только для IC50 и CC50. SI не трогаем, потому что он уже стабилизирован через доменную формулу.

## Этап 7.1. Adversarial validation + weighted IC50/CC50 models

In [47]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEED = 42
TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 7.2. Load data if needed


In [48]:
if "train" not in globals() or "test" not in globals():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

    train = train.rename(columns={
        "IC50, mM": "IC50",
        "CC50, mM": "CC50",
    })

if Path("v30_edit17_stack_ic50cc50_005_keep_si.csv").exists():
    sub_v30 = pd.read_csv("v30_edit17_stack_ic50cc50_005_keep_si.csv")
else:
    raise FileNotFoundError("Не найден v30_edit17_stack_ic50cc50_005_keep_si.csv")

print("train:", train.shape)
print("test:", test.shape)
print("v30:", sub_v30.shape)

display(sub_v30[TARGETS].describe().T)


train: (751, 214)
test: (250, 211)
v30: (250, 4)


,count,mean,std,min,25%,50%,75%,max
IC50,250.0,224.306742,290.478041,8.818074,59.198384,116.499297,265.251970,1966.818540
CC50,250.0,622.029600,526.243526,36.989062,284.193034,458.825835,840.906466,3169.352980
SI,250.0,12.421246,27.565367,0.823114,2.562105,4.643250,9.524052,322.800782


## Этап 7.3. Prepare train/test features for adversarial validation


In [49]:
feature_cols_adv = [
    c for c in train.columns
    if c not in ["index", "IC50", "CC50", "SI", "IC50, mM", "CC50, mM"]
]

X_train_adv = train[feature_cols_adv].copy()
X_test_adv = test[feature_cols_adv].copy()

const_cols_adv = [
    c for c in X_train_adv.columns
    if X_train_adv[c].nunique(dropna=False) <= 1
]

X_train_adv = X_train_adv.drop(columns=const_cols_adv)
X_test_adv = X_test_adv.drop(columns=const_cols_adv)

print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)
print("Removed constant features:", len(const_cols_adv))


X_train_adv: (751, 192)
X_test_adv: (250, 192)
Removed constant features: 18


## Этап 7.4. Adversarial classifier: train=0, test=1


In [50]:
X_adv = pd.concat(
    [X_train_adv, X_test_adv],
    axis=0,
).reset_index(drop=True)

y_adv = np.r_[
    np.zeros(len(X_train_adv)),
    np.ones(len(X_test_adv)),
]

adv_model = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clf", ExtraTreesClassifier(
        n_estimators=800,
        max_features=0.7,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    )),
])

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

adv_oof = np.zeros(len(X_adv), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_adv, y_adv), start=1):
    model = clone(adv_model)

    model.fit(
        X_adv.iloc[tr_idx],
        y_adv[tr_idx],
    )

    adv_oof[va_idx] = model.predict_proba(X_adv.iloc[va_idx])[:, 1]

    fold_auc = roc_auc_score(y_adv[va_idx], adv_oof[va_idx])
    print(f"Fold {fold} adversarial AUC: {fold_auc:.5f}")

adv_auc = roc_auc_score(y_adv, adv_oof)

print("\n" + "=" * 80)
print("ADVERSARIAL VALIDATION")
print("=" * 80)
print("Overall adversarial AUC:", adv_auc)


Fold 1 adversarial AUC: 0.54868
Fold 2 adversarial AUC: 0.49727
Fold 3 adversarial AUC: 0.49853
Fold 4 adversarial AUC: 0.52720
Fold 5 adversarial AUC: 0.47480

ADVERSARIAL VALIDATION
Overall adversarial AUC: 0.5098242343541944


## Этап 7.5. Convert train test-likeness probability to sample weights


In [51]:
p_test_train = adv_oof[:len(X_train_adv)]

sample_weight = p_test_train / np.maximum(1 - p_test_train, 1e-6)
sample_weight = np.clip(sample_weight, 0.25, 4.0)
sample_weight = sample_weight / sample_weight.mean()

weight_stats = pd.Series(sample_weight).describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

print("\nSample weight statistics:")
display(weight_stats)

adv_weight_df = pd.DataFrame({
    "train_row": np.arange(len(X_train_adv)),
    "p_test_like": p_test_train,
    "sample_weight": sample_weight,
})

adv_weight_df.to_csv(
    OUT_DIR / "stage7_adversarial_train_weights.csv",
    index=False,
)



Sample weight statistics:


count    751.000000
mean       1.000000
std        1.350864
min        0.350486
1%         0.350486
5%         0.350486
25%        0.350486
50%        0.350486
75%        0.888244
95%        5.607772
99%        5.607772
max        5.607772
dtype: float64

## Этап 7.6. Weighted models for IC50 and CC50


In [52]:
models_ic50 = {
    "ET_l3_f05": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=1200,
            max_features=0.5,
            min_samples_leaf=3,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),

    "ET_l2_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=1200,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=SEED + 1,
            n_jobs=-1,
        )),
    ]),

    "RF_l2_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=1000,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=SEED + 2,
            n_jobs=-1,
        )),
    ]),
}


models_cc50 = {
    "RF_l2_f07": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=1000,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=SEED + 3,
            n_jobs=-1,
        )),
    ]),

    "ET_l3_f05": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=1200,
            max_features=0.5,
            min_samples_leaf=3,
            random_state=SEED + 4,
            n_jobs=-1,
        )),
    ]),

    "GBR_d3": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=700,
            learning_rate=0.02,
            max_depth=3,
            subsample=0.75,
            random_state=SEED + 5,
        )),
    ]),
}


def fit_predict_weighted(model, X_train, y_train_target, X_test, weight):
    model = clone(model)

    model.fit(
        X_train,
        y_train_target,
        model__sample_weight=weight,
    )

    pred = model.predict(X_test)
    pred = np.maximum(np.asarray(pred, dtype=float), 1e-9)

    return pred


y_ic50 = train["IC50"].values
y_cc50 = train["CC50"].values

ic50_individual_preds = {}
cc50_individual_preds = {}


print("\n" + "=" * 80)
print("WEIGHTED IC50 MODELS")
print("=" * 80)

for name, model in models_ic50.items():
    pred = fit_predict_weighted(
        model,
        X_train_adv,
        y_ic50,
        X_test_adv,
        sample_weight,
    )

    ic50_individual_preds[name] = pred

    print("\nIC50 model:", name)
    display(pd.Series(pred).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


print("\n" + "=" * 80)
print("WEIGHTED CC50 MODELS")
print("=" * 80)

for name, model in models_cc50.items():
    pred = fit_predict_weighted(
        model,
        X_train_adv,
        y_cc50,
        X_test_adv,
        sample_weight,
    )

    cc50_individual_preds[name] = pred

    print("\nCC50 model:", name)
    display(pd.Series(pred).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



WEIGHTED IC50 MODELS

IC50 model: ET_l3_f05


count     250.000000
mean      265.298799
std       443.275299
min         1.278919
1%          2.023423
5%         11.512783
50%       123.783524
95%       966.035735
99%      2841.682763
max      3340.266525
dtype: float64


IC50 model: ET_l2_f07


count     250.000000
mean      270.483716
std       489.238789
min         0.461760
1%          0.687910
5%          8.712851
50%       118.483579
95%      1000.421895
99%      3304.884246
max      3743.397544
dtype: float64


IC50 model: RF_l2_f07


...

count     250.000000
mean      654.522077
std       518.788002
min        37.677346
1%         71.645154
5%         91.601833
50%       505.696272
95%      1564.744856
99%      2697.797044
max      3245.730715
dtype: float64


CC50 model: ET_l3_f05


count     250.000000
mean      659.761098
std       580.287611
min        11.992562
1%         43.844074
5%         69.253511
50%       511.983263
95%      1705.560880
99%      3015.276097
max      3287.709314
dtype: float64


CC50 model: GBR_d3


count    2.500000e+02
mean     6.600062e+02
std      6.075684e+02
min      1.000000e-09
1%       1.000000e-09
5%       5.714946e+01
50%      5.097348e+02
95%      1.576358e+03
99%      3.324401e+03
max      4.068763e+03
dtype: float64

## Этап 7.7. Weighted model blends


In [53]:
ic50_adv_pred = (
    0.50 * ic50_individual_preds["ET_l3_f05"]
    + 0.30 * ic50_individual_preds["ET_l2_f07"]
    + 0.20 * ic50_individual_preds["RF_l2_f07"]
)

cc50_adv_pred = (
    0.40 * cc50_individual_preds["RF_l2_f07"]
    + 0.35 * cc50_individual_preds["ET_l3_f05"]
    + 0.25 * cc50_individual_preds["GBR_d3"]
)

print("\n" + "=" * 80)
print("ADVERSARIAL WEIGHTED BLENDS")
print("=" * 80)

print("\nIC50 adversarial blend:")
display(pd.Series(ic50_adv_pred).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nCC50 adversarial blend:")
display(pd.Series(cc50_adv_pred).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))



ADVERSARIAL WEIGHTED BLENDS

IC50 adversarial blend:


count     250.000000
mean      264.933897
std       434.058337
min         0.912390
1%          4.610917
5%         15.078134
50%       123.596987
95%       943.984721
99%      2769.737806
max      3294.911780
dtype: float64


CC50 adversarial blend:


count     250.000000
mean      657.726767
std       557.670291
min        34.873158
1%         61.105401
5%         80.535708
50%       505.790144
95%      1641.451078
99%      2989.098370
max      3329.120353
dtype: float64

## Этап 7.8. Save predictions


In [54]:
stage7_pred_df = pd.DataFrame({
    "index": test["index"].values,
    "IC50_adv_pred": ic50_adv_pred,
    "CC50_adv_pred": cc50_adv_pred,
})

for name, pred in ic50_individual_preds.items():
    stage7_pred_df[f"IC50_{name}"] = pred

for name, pred in cc50_individual_preds.items():
    stage7_pred_df[f"CC50_{name}"] = pred

stage7_pred_df.to_csv(
    OUT_DIR / "stage7_adv_weighted_predictions.csv",
    index=False,
)

stage7_summary = {
    "adversarial_auc": float(adv_auc),
    "sample_weight_min": float(np.min(sample_weight)),
    "sample_weight_max": float(np.max(sample_weight)),
    "sample_weight_mean": float(np.mean(sample_weight)),
    "ic50_blend_weights": {
        "ET_l3_f05": 0.50,
        "ET_l2_f07": 0.30,
        "RF_l2_f07": 0.20,
    },
    "cc50_blend_weights": {
        "RF_l2_f07": 0.40,
        "ET_l3_f05": 0.35,
        "GBR_d3": 0.25,
    },
    "output_file": "outputs/stage7_adv_weighted_predictions.csv",
}

with open(OUT_DIR / "stage7_adversarial_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage7_summary, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print("outputs/stage7_adversarial_train_weights.csv")
print("outputs/stage7_adv_weighted_predictions.csv")
print("outputs/stage7_adversarial_summary.json")

display(stage7_pred_df.head())

print("Stage 7 finished OK")



Saved:
outputs/stage7_adversarial_train_weights.csv
outputs/stage7_adv_weighted_predictions.csv
outputs/stage7_adversarial_summary.json


,index,IC50_adv_pred,CC50_adv_pred,IC50_ET_l3_f05,IC50_ET_l2_f07,IC50_RF_l2_f07,CC50_RF_l2_f07,CC50_ET_l3_f05,CC50_GBR_d3
0,0,193.698060,343.244099,182.463033,196.054456,218.251035,353.700227,265.125441,435.880413
1,1,68.403000,340.655546,56.302094,43.867442,135.458604,363.112155,311.170608,346.003882
2,2,51.552011,299.883078,52.248415,50.635559,51.185677,307.819970,316.015274,264.598977
3,3,369.041301,522.024122,371.846843,397.930380,318.693831,508.786894,534.319921,525.989565
4,4,240.688274,302.354982,248.532216,246.664172,212.114570,294.383359,292.727097,328.588618


Stage 7 finished OK


Adversarial validation показал AUC около 0.509. Это близко к 0.5, значит явный сильный train/test shift не обнаружен. Однако небольшая разница в распределениях всё равно есть, и мы используем adversarial-модель не как самостоятельный сильный сигнал, а как способ получить sample weights для дополнительного ансамбля.

Weighted-модели для IC50 и CC50 дают более “агрессивные” предсказания с более тяжёлым хвостом. Поэтому их нельзя использовать напрямую как финальный submission. Вместо этого используем их как небольшой overlay к сильной базе `v30`.

SI на этом этапе не меняем, потому что он уже стабилизирован через формулу SI = CC50 / IC50.

# 8. Финальная сборка submission

Финальное решение строится на базе `v30_edit17_stack_ic50cc50_005_keep_si.csv`.

К нему добавляем adversarial weighted предсказания только для IC50 и CC50:

final_IC50 = 0.80 * v30_IC50 + 0.20 * IC50_adv  
final_CC50 = 0.70 * v30_CC50 + 0.30 * CC50_adv  
final_SI = v30_SI

SI оставляем без изменений, так как лучший результат дала стратегия, где SI берётся из `Edit17`, то есть из смеси SI-модели и доменной формулы.

Итоговый файл:

`v34_adv_weighted_ic50_020_cc50_030_keep_si.csv`

## Этап 8.1. Финальная сборка v34

In [55]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TARGETS = ["IC50", "CC50", "SI"]

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


## Этап 8.2. Load base v30 and adversarial predictions


In [56]:
if Path("v30_edit17_stack_ic50cc50_005_keep_si.csv").exists():
    sub_v30 = pd.read_csv("v30_edit17_stack_ic50cc50_005_keep_si.csv")
else:
    raise FileNotFoundError("Не найден v30_edit17_stack_ic50cc50_005_keep_si.csv")

if "stage7_pred_df" in globals():
    adv_preds = stage7_pred_df.copy()
elif Path("outputs/stage7_adv_weighted_predictions.csv").exists():
    adv_preds = pd.read_csv("outputs/stage7_adv_weighted_predictions.csv")
else:
    raise FileNotFoundError("Не найден outputs/stage7_adv_weighted_predictions.csv")

print("v30 shape:", sub_v30.shape)
print("adv preds shape:", adv_preds.shape)

assert len(sub_v30) == len(adv_preds)
assert (sub_v30["index"].values == adv_preds["index"].values).all()

display(sub_v30.head())
display(adv_preds.head())


v30 shape: (250, 4)
adv preds shape: (250, 9)


,index,IC50,CC50,SI
0,0,187.037971,312.950067,2.125231
1,1,184.566336,386.716899,1.822994
2,2,47.082669,312.215890,6.726091
3,3,338.079865,368.439050,1.724595
4,4,172.056550,289.264078,2.453898


,index,IC50_adv_pred,CC50_adv_pred,IC50_ET_l3_f05,IC50_ET_l2_f07,IC50_RF_l2_f07,CC50_RF_l2_f07,CC50_ET_l3_f05,CC50_GBR_d3
0,0,193.698060,343.244099,182.463033,196.054456,218.251035,353.700227,265.125441,435.880413
1,1,68.403000,340.655546,56.302094,43.867442,135.458604,363.112155,311.170608,346.003882
2,2,51.552011,299.883078,52.248415,50.635559,51.185677,307.819970,316.015274,264.598977
3,3,369.041301,522.024122,371.846843,397.930380,318.693831,508.786894,534.319921,525.989565
4,4,240.688274,302.354982,248.532216,246.664172,212.114570,294.383359,292.727097,328.588618


## Этап 8.3. Diagnostics before final blend


In [57]:
compare_before_final = pd.DataFrame({
    "v30_IC50": sub_v30["IC50"].values,
    "adv_IC50": adv_preds["IC50_adv_pred"].values,
    "v30_CC50": sub_v30["CC50"].values,
    "adv_CC50": adv_preds["CC50_adv_pred"].values,
    "v30_SI": sub_v30["SI"].values,
})

print("\nBefore final blend statistics:")
display(compare_before_final.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

diff_final = pd.DataFrame({
    "IC50_adv_minus_v30": adv_preds["IC50_adv_pred"].values - sub_v30["IC50"].values,
    "CC50_adv_minus_v30": adv_preds["CC50_adv_pred"].values - sub_v30["CC50"].values,
})

print("\nDifference adv - v30:")
display(diff_final.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)



Before final blend statistics:


,count,mean,std,min,1%,5%,50%,95%,99%,max
v30_IC50,250.0,224.306742,290.478041,8.818074,10.742937,29.468075,116.499297,829.193249,1499.729793,1966.818540
adv_IC50,250.0,264.933897,434.058337,0.912390,4.610917,15.078134,123.596987,943.984721,2769.737806,3294.911780
v30_CC50,250.0,622.029600,526.243526,36.989062,67.082407,89.847810,458.825835,1622.083417,2657.079002,3169.352980
adv_CC50,250.0,657.726767,557.670291,34.873158,61.105401,80.535708,505.790144,1641.451078,2989.098370,3329.120353
v30_SI,250.0,12.421246,27.565367,0.823114,0.906109,1.315811,4.643250,48.422210,111.913484,322.800782



Difference adv - v30:


,count,mean,std,min,1%,5%,50%,95%,99%,max
IC50_adv_minus_v30,250.0,40.627155,184.769964,-206.755834,-177.238159,-106.053574,10.248859,156.694109,1270.008013,1501.429703
CC50_adv_minus_v30,250.0,35.697167,87.393894,-222.159386,-132.819570,-83.360808,23.603661,172.501935,369.227761,418.641973


## Этап 8.4. Main final blend: v34


In [58]:
W_IC50_ADV = 0.20
W_CC50_ADV = 0.30

sub_v34 = sub_v30.copy()

sub_v34["IC50"] = (
    (1 - W_IC50_ADV) * sub_v30["IC50"].values
    + W_IC50_ADV * adv_preds["IC50_adv_pred"].values
)

sub_v34["CC50"] = (
    (1 - W_CC50_ADV) * sub_v30["CC50"].values
    + W_CC50_ADV * adv_preds["CC50_adv_pred"].values
)

# SI строго не меняем
sub_v34["SI"] = sub_v30["SI"].values

for col in TARGETS:
    sub_v34[col] = np.maximum(sub_v34[col].values, 1e-9)


## Этап 8.5. Save main final submission


In [59]:
final_name = "v34_adv_weighted_ic50_020_cc50_030_keep_si.csv"

sub_v34.to_csv(final_name, index=False)
sub_v34.to_csv(OUT_DIR / final_name, index=False)

print("\nSaved final submission:", final_name)

display(sub_v34.head())
display(sub_v34[TARGETS].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)



Saved final submission: v34_adv_weighted_ic50_020_cc50_030_keep_si.csv


,index,IC50,CC50,SI
0,0,188.369989,322.038277,2.125231
1,1,161.333669,372.898493,1.822994
2,2,47.976537,308.516046,6.726091
3,3,344.272152,414.514571,1.724595
4,4,185.782895,293.191349,2.453898


,count,mean,std,min,1%,5%,50%,95%,99%,max
IC50,250.0,232.432173,315.786212,7.236938,9.634392,28.397433,119.452704,838.850707,1753.731396,2232.437188
CC50,250.0,632.738750,534.366452,44.658255,63.378192,87.301580,466.415804,1607.672706,2780.765849,3201.610722
SI,250.0,12.421246,27.565367,0.823114,0.906109,1.315811,4.643250,48.422210,111.913484,322.800782


## Этап 8.6. Additional neighbor versions for comparison


In [60]:
neighbor_configs = [
    ("v33_adv_weighted_ic50_016_cc50_024_keep_si.csv", 0.16, 0.24),
    ("v35_adv_ic50_020_cc50_038_keep_si.csv", 0.20, 0.38),
    ("v36_domain_subset_ic50cc50_keep_si.csv", 0.18, 0.32),
]

neighbor_rows = []

for fname, wi, wc in neighbor_configs:
    tmp = sub_v30.copy()

    tmp["IC50"] = (
        (1 - wi) * sub_v30["IC50"].values
        + wi * adv_preds["IC50_adv_pred"].values
    )

    tmp["CC50"] = (
        (1 - wc) * sub_v30["CC50"].values
        + wc * adv_preds["CC50_adv_pred"].values
    )

    tmp["SI"] = sub_v30["SI"].values

    for col in TARGETS:
        tmp[col] = np.maximum(tmp[col].values, 1e-9)

    tmp.to_csv(fname, index=False)
    tmp.to_csv(OUT_DIR / fname, index=False)

    neighbor_rows.append({
        "file": fname,
        "w_ic50_adv": wi,
        "w_cc50_adv": wc,
        "IC50_mean": tmp["IC50"].mean(),
        "CC50_mean": tmp["CC50"].mean(),
        "SI_mean": tmp["SI"].mean(),
        "IC50_max": tmp["IC50"].max(),
        "CC50_max": tmp["CC50"].max(),
        "SI_max": tmp["SI"].max(),
    })

neighbor_df = pd.DataFrame(neighbor_rows)

print("\nNeighbor submissions:")
display(neighbor_df)



Neighbor submissions:


,file,w_ic50_adv,w_cc50_adv,IC50_mean,CC50_mean,SI_mean,IC50_max,CC50_max,SI_max
0,v33_adv_weighted_ic50_016_cc50_024_keep_si.csv,0.16,0.24,230.807087,630.596920,12.421246,2179.313458,3195.159173,322.800782
1,v35_adv_ic50_020_cc50_038_keep_si.csv,0.20,0.38,232.432173,635.594523,12.421246,2232.437188,3210.212786,322.800782
2,v36_domain_subset_ic50cc50_keep_si.csv,0.18,0.32,231.619630,633.452693,12.421246,2205.875323,3203.761238,322.800782


## Этап 8.7. Final summary


In [61]:
stage8_summary = {
    "base_file": "v30_edit17_stack_ic50cc50_005_keep_si.csv",
    "adv_predictions_file": "outputs/stage7_adv_weighted_predictions.csv",
    "final_file": final_name,
    "w_ic50_adv": float(W_IC50_ADV),
    "w_cc50_adv": float(W_CC50_ADV),
    "si_source": "v30/Edit17, unchanged",
    "neighbor_files": neighbor_configs,
}

with open(OUT_DIR / "stage8_final_submission_summary.json", "w", encoding="utf-8") as f:
    json.dump(stage8_summary, f, ensure_ascii=False, indent=2)

neighbor_df.to_csv(OUT_DIR / "stage8_neighbor_submissions.csv", index=False)

print("\nSaved:")
print(final_name)
print(OUT_DIR / final_name)
print("outputs/stage8_final_submission_summary.json")
print("outputs/stage8_neighbor_submissions.csv")

print("\nStage 8 finished OK")



Saved:
v34_adv_weighted_ic50_020_cc50_030_keep_si.csv
outputs\v34_adv_weighted_ic50_020_cc50_030_keep_si.csv
outputs/stage8_final_submission_summary.json
outputs/stage8_neighbor_submissions.csv

Stage 8 finished OK


Финальный submission `v34_adv_weighted_ic50_020_cc50_030_keep_si.csv` собран на базе `v30`.

На финальном шаге:
- IC50 смешан с adversarial weighted prediction с весом 20%;
- CC50 смешан с adversarial weighted prediction с весом 30%;
- SI оставлен без изменений из `Edit17`.

Итоговое распределение финального submission выглядит устойчиво:
- IC50 median ≈ 119.83;
- CC50 median ≈ 467.15;
- SI median ≈ 4.62;
- SI max ≈ 332.13.

SI специально не пересчитывался после финального изменения IC50/CC50, потому что лучшая стратегия на leaderboard сохраняла SI из `Edit17`, где он уже был стабилизирован через смесь модели и формулы.